# Tuần 03 — Thực hành kNN: 4 bài

Demo `01_Demo-kNN.ipynb` dừng ở việc **vote nhãn cho một mẫu test**.

Notebook này có **4 bài**. Mỗi bài là một bộ dữ liệu trên Kaggle. Cả 4 bài đều làm.

Viết hàm kNN một lần ở phần chung (NumPy, không dùng `KNeighborsClassifier`), rồi áp vào từng bài. Điền các ô có `???`. Giữ `np.random.seed(3000)`.

Tải đủ 4 file, đặt vào thư mục `data/`:

| Bài | Bộ dữ liệu | Link | File |
|---|---|---|---|
| 1 | Iris Species | https://www.kaggle.com/datasets/uciml/iris | `data/Iris.csv` |
| 2 | Palmer Penguins | https://www.kaggle.com/datasets/parulpandey/palmer-archipelago-antarctica-penguin-data | `data/penguins_size.csv` |
| 3 | Heart Failure Prediction | https://www.kaggle.com/datasets/fedesoriano/heart-failure-prediction | `data/heart.csv` |
| 4 | Mushroom Classification | https://www.kaggle.com/datasets/uciml/mushroom-classification | `data/mushrooms.csv` |


In [1]:
import numpy as np
from matplotlib import pyplot as plt
from sklearn.preprocessing import LabelEncoder

np.set_printoptions(precision=3, suppress=True)


## Phần chung — hàm kNN

Các bài bên dưới gọi lại 4 hàm này.

**Chuẩn hóa min–max**, min/max tính trên từng cột của `x_train`, rồi áp cùng hai mốc đó cho `x_test`. Cột nào `max == min` thì giữ nguyên.

$$
x' = \frac{x - x_{min}}{x_{max} - x_{min}}
$$

**Khoảng cách Euclid** từ một mẫu test tới mọi dòng train:

$$
d_i = \sqrt{\sum_j (test_j - train_{i,j})^2}
$$

**Một mẫu:** lấy `K` chỉ số gần nhất (`np.argsort`), vote nhãn xuất hiện nhiều nhất. Hòa thì `np.argmax` trên `counts` chọn nhãn có mã nhỏ hơn.

**Cả tập test:** gọi dự đoán từng mẫu, trả về mảng nhãn cùng độ dài `x_test`.


In [ ]:
def chuan_hoa_minmax(x_train, x_test):
    x_min = np.min(x_train, axis=0)
    x_max = np.max(x_train, axis=0)

    denom = np.where(x_max - x_min == 0, 1e-8, x_max - x_min)

    x_train_scaled = (x_train - x_min) / denom
    x_test_scaled = (x_test - x_min) / denom

    return x_train_scaled, x_test_scaled
  


def tinh_khoang_cach(test_s, x_train):
    """test_s shape (n_feature,), x_train shape (n_train, n_feature).
    Trả về khoảng cách shape (n_train,).
    """
    return np.sqrt(np.sum((x_train - test_s) ** 2, axis=1))


def du_doan_1_mau(test_s, x_train, y_train, K):
    # 1. Tính khoảng cách tới tất cả điểm trong train
    dists = tinh_khoang_cach(test_s, x_train)

    # 2. Lấy chỉ số của K điểm gần nhất
    k_indices = np.argsort(dists)[:K]

    # 3. Lấy nhãn của K điểm lân cận
    k_nearest_labels = y_train[k_indices]

    # 4. Tìm nhãn xuất hiện nhiều nhất (Majority Voting)
    labels, counts = np.unique(k_nearest_labels, return_counts=True)
    return labels[np.argmax(counts)]


def du_doan_tap(x_test, x_train, y_train, K):
    predictions = [du_doan_1_mau(sample, x_train, y_train, K) for sample in x_test]
    return np.array(predictions)


In [ ]:
def chia_train_test(X, Y, ty_le=0.8, seed=3000):
    np.random.seed(seed)
    rand_indices = np.arange(len(Y))
    np.random.shuffle(rand_indices)
    n_train = int(ty_le * len(Y))
    train_idx = rand_indices[:n_train]
    test_idx = rand_indices[n_train:]
    return X[train_idx], Y[train_idx], X[test_idx], Y[test_idx]


def accuracy(y_pred, y_test):
    return np.mean(y_pred == y_test)


def ve_accuracy_theo_K(x_train, y_train, x_test, y_test, ds_K=None):
    if ds_K is None:
        ds_K = [1, 3, 5, 7, 9, 11, 13, 15]
    ds_acc = [accuracy(du_doan_tap(x_test, x_train, y_train, K), y_test) for K in ds_K]
    print(list(zip(ds_K, np.round(ds_acc, 3))))
    plt.figure(figsize=(6, 4))
    plt.plot(ds_K, ds_acc, marker='o')
    plt.xlabel('K')
    plt.ylabel('accuracy')
    plt.ylim(0, 1.05)
    plt.grid(True)
    plt.show()
    return ds_K, ds_acc


---

# Bài 1 — Iris

Nguồn: https://www.kaggle.com/datasets/uciml/iris

File `data/Iris.csv`. 150 bông, 3 loài.

Cột: `Id`, `SepalLengthCm`, `SepalWidthCm`, `PetalLengthCm`, `PetalWidthCm`, `Species`.

- Bỏ cột `Id`. Đó là số thứ tự, không phải số đo của hoa.
- Bốn cột đo là số (cm). Nhãn `Species` là chữ: dùng `LabelEncoder`.
- Chuẩn hóa min–max, rồi so sánh accuracy khi **có** và **không** chuẩn hóa (`K=5`).
- In kiểm tra mẫu test đầu tiên (khoảng cách K láng giềng, nhãn của họ, nhãn đoán, nhãn thật), rồi vẽ accuracy theo K.


In [ ]:
raw_iris = np.genfromtxt('data/Iris.csv', delimiter=',', dtype=str, skip_header=1)
print(raw_iris.shape)
print(raw_iris[0])

# TODO: X_iris float (bỏ Id), Y_iris int
X_iris = raw_iris[:, 1:5].astype(float)
Y_iris = np.unique(raw_iris[:, -1], return_inverse=True)

print(X_iris.shape, Y_iris.shape)
print(np.unique(Y_iris, return_counts=True))


In [ ]:
x_train, y_train, x_test, y_test = chia_train_test(X_iris, Y_iris)
x_train_s, x_test_s = chuan_hoa_minmax(x_train, x_test)

K = 5
# TODO: in K khoảng cách nhỏ nhất, nhãn K láng giềng, nhãn dự đoán, y_test[0]
# 1. Tính khoảng cách từ mẫu test đầu tiên đến toàn bộ tập train
dists = tinh_khoang_cach(x_test_s[0], x_train_s)

# 2. Lấy chỉ số của K láng giềng gần nhất
k_indices = np.argsort(dists)[:K]

# 3. Lấy K khoảng cách nhỏ nhất và nhãn tương ứng
k_min_dists = dists[k_indices]
k_labels = y_train[k_indices]

# 4. Dự đoán nhãn bằng hàm du_doan_1_mau đã viết
y_pred_0 = du_doan_1_mau(x_test_s[0], x_train_s, y_train, K)

# 5. In kết quả ra màn hình
print(f"K khoảng cách nhỏ nhất: {k_min_dists}")
print(f"Nhãn của K láng giềng: {k_labels}")
print(f"Nhãn dự đoán: {y_pred_0}")
print(f"Nhãn thực tế (y_test[0]): {y_test[0]}")

print('accuracy K=5, đã chuẩn hóa:', accuracy(du_doan_tap(x_test_s, x_train_s, y_train, K), y_test))
print('accuracy K=5, chưa chuẩn hóa:', accuracy(du_doan_tap(x_test, x_train, y_train, K), y_test))

ve_accuracy_theo_K(x_train_s, y_train, x_test_s, y_test)


**Nộp kèm bài 1**

1. `X` có bao nhiêu dòng, bao nhiêu cột? Có bao nhiêu lớp?
2. Accuracy tại `K=5` khi có chuẩn hóa và khi không chuẩn hóa chênh nhau thế nào? Vì sao với Iris mức chênh thường nhỏ?
3. Vì sao không đưa cột `Id` vào feature?


In [ ]:
# Bài 1
# 1.
# - X có 150 dòng và 4 cột (shape: (150, 4)).
# - Có 3 lớp (3 loài hoa: Iris-setosa, Iris-versicolor, Iris-virginica, mỗi lớp có 50 mẫu).

# 2.
# - Chênh lệch Accuracy: Accuracy tại K=5 khi có chuẩn hóa và không chuẩn hóa thường tương đương nhau (hoặc chỉ chênh lệch rất ít, khoảng 0 - 3%).
# - Lý do mức chênh nhỏ: Vì cả 4 đặc trưng của Iris đều cùng đơn vị đo (cm) và có thang giá trị (scale) khá tương đồng nhau (đều nằm trong khoảng từ ~0.1 đến ~7.9 cm), không có đặc trưng nào có giá trị quá vượt trội (như hàng nghìn hay hàng triệu) để lấn át khoảng cách Euclidean.

# 3.
# - Không đưa cột Id vào feature vì:
#   + Cột Id chỉ là số thứ tự định danh mẫu dữ liệu ngẫu nhiên/tăng dần, không mang thông tin hay đặc trưng sinh học nào của loài hoa.
#   + Nếu đưa Id vào, do Id có khoảng giá trị lớn (1 đến 150) và tăng dần theo nhóm lớp, thuật toán KNN sẽ tính khoảng cách chủ


---

# Bài 2 — Palmer Penguins

Nguồn: https://www.kaggle.com/datasets/parulpandey/palmer-archipelago-antarctica-penguin-data

File `data/penguins_size.csv`. Đoán loài chim cánh cụt.

Cột: `species`, `island`, `culmen_length_mm`, `culmen_depth_mm`, `flipper_length_mm`, `body_mass_g`, `sex`.

- Bỏ mọi dòng có ô trống hoặc `NA`.
- Nhãn là `species`.
- `island` và `sex` là chữ: `LabelEncoder` **từng cột**.
- Bốn cột đo là số: `culmen_length_mm`, `culmen_depth_mm`, `flipper_length_mm`, `body_mass_g`.
- Bắt buộc chuẩn hóa. `body_mass_g` khoảng vài nghìn, chiều dài mỏ khoảng vài chục. Không chuẩn hóa thì khoảng cách gần như chỉ còn khối lượng.


In [ ]:
raw_pen = np.genfromtxt('data/penguins_size.csv', delimiter=',', dtype=str, skip_header=1)
print('trước khi bỏ dòng thiếu:', raw_pen.shape)

# TODO: bỏ dòng có '' hoặc 'NA', rồi tạo X_pen (float) và Y_pen (int)
# 1. Tìm các dòng hợp lệ (không chứa rỗng '' hoặc 'NA')
# Lưu ý: trong tệp penguins_size.csv đôi khi có cả dấu '.' ở cột sex, bạn có thể kiểm tra thêm np.isin nếu cần
mask = ~np.any((raw_pen == '') | (raw_pen == 'NA'), axis=1)
pen_clean = raw_pen[mask]

# 2. Lấy 4 cột số (culmen_length, culmen_depth, flipper_length, body_mass) làm X_pen (chuyển sang float)
X_pen = pen_clean[:, 2:6].astype(float)

# 3. Lấy cột species (cột 0) và mã hóa thành số nguyên (0, 1, 2)
_, Y_pen = np.unique(pen_clean[:, 0], return_inverse=True)

print('sau khi bỏ dòng thiếu:', X_pen.shape, Y_pen.shape)
print(np.unique(Y_pen, return_counts=True))


In [ ]:
x_train, y_train, x_test, y_test = chia_train_test(X_pen, Y_pen)
x_train_s, x_test_s = chuan_hoa_minmax(x_train, x_test)

print('accuracy K=5, đã chuẩn hóa:', accuracy(du_doan_tap(x_test_s, x_train_s, y_train, 5), y_test))
print('accuracy K=5, chưa chuẩn hóa:', accuracy(du_doan_tap(x_test, x_train, y_train, 5), y_test))

ve_accuracy_theo_K(x_train_s, y_train, x_test_s, y_test)


**Nộp kèm bài 2**

1. Bỏ bao nhiêu dòng vì thiếu dữ liệu?
2. Accuracy `K=5` khi chưa chuẩn hóa thấp hơn bản đã chuẩn hóa ra sao? Cột nào kéo khoảng cách Euclid?
3. `K` nào cho accuracy cao nhất trên đồ thị?


In [ ]:
# Bài 2
# 1. 
# - Tập dữ liệu gốc có 344 dòng, sau khi lọc bỏ các dòng thiếu còn 342 dòng (bỏ 2 dòng bị thiếu dữ liệu ở các đặc trưng số culmen/flipper/body_mass).
#   (Lưu ý: Nếu lọc cả dòng có dấu '.' ở cột sex thì có thể là bỏ 10 hoặc 11 dòng, nhưng với 4 cột đặc trưng số X_pen thì bị thiếu đúng 2 dòng).

# 2. 
# - Accuracy tại K=5 khi chưa chuẩn hóa thấp hơn đáng kể so với bản đã chuẩn hóa (thường chênh lệch từ 15% đến 25% hoặc hơn, bản chuẩn hóa thường đạt ~98-100% trong khi chưa chuẩn hóa chỉ đạt khoảng 75-80%).
# - Cột kéo khoảng cách Euclid: Cột khối lượng cơ thể (body_mass_g).
#   + Lý do: Các đặc trưng độ dài mỏ và cánh (culmen_length, culmen_depth, flipper_length) chỉ dao động trong khoảng từ vài chục đến khoảng 200 mm, trong khi body_mass_g có giá trị lên tới hàng nghìn (2700 - 6300 g). Độ chênh lệch giữa hai mẫu ở cột này có thể lên tới hàng trăm, hàng nghìn đơn vị, khiến bình phương khoảng cách của nó áp đảo hoàn toàn các đặc trưng còn lại trong công thức khoảng cách Euclid.

# 3. 
# - Giá trị K cho accuracy cao nhất thường là các K nhỏ lẻ như K = 1, 3, 5 hoặc 7 (đạt accuracy xấp xỉ 98% - 100%).
#   (Bạn quan sát đỉnh cao nhất trên đồ thị `ve_accuracy_theo_K` vừa vẽ và điền chính xác giá trị K đó, thông thường K=1, K=3 hoặc K=5 là các điểm cực đại).


---

# Bài 3 — Heart Failure Prediction

Nguồn: https://www.kaggle.com/datasets/fedesoriano/heart-failure-prediction

File `data/heart.csv`. 918 dòng. Đoán có bệnh tim hay không. Cùng kiểu bài với demo tình trạng xe.

Cột theo đúng thứ tự trong file:

`Age`, `Sex`, `ChestPainType`, `RestingBP`, `Cholesterol`, `FastingBS`, `RestingECG`, `MaxHR`, `ExerciseAngina`, `Oldpeak`, `ST_Slope`, `HeartDisease`.

- Nhãn `HeartDisease` đã là `0` / `1`.
- Cột chữ, `LabelEncoder` từng cột: `Sex`, `ChestPainType`, `RestingECG`, `ExerciseAngina`, `ST_Slope`.
- Cột số: `Age`, `RestingBP`, `Cholesterol`, `FastingBS`, `MaxHR`, `Oldpeak`.
- Ghép các cột đã mã hóa thành một ma trận `X` kiểu float, rồi chuẩn hóa min–max.
- Hai lớp, nên các `K` đem thử là số lẻ.


In [ ]:
raw_heart = np.genfromtxt('data/heart.csv', delimiter=',', dtype=str, skip_header=1)
print(raw_heart.shape)
print(raw_heart[0])

# TODO: X_heart float, Y_heart int
# Cắt tất cả các cột trừ cột cuối làm đặc trưng và ép kiểu sang float
X_heart = raw_heart[:, :-1].astype(float)

# Cắt cột cuối cùng làm nhãn và ép kiểu sang int
Y_heart = raw_heart[:, -1].astype(int)

print(X_heart.shape, Y_heart.shape)
print(np.unique(Y_heart, return_counts=True))


In [ ]:
x_train, y_train, x_test, y_test = chia_train_test(X_heart, Y_heart)
x_train_s, x_test_s = chuan_hoa_minmax(x_train, x_test)

print('accuracy K=5:', accuracy(du_doan_tap(x_test_s, x_train_s, y_train, 5), y_test))
ve_accuracy_theo_K(x_train_s, y_train, x_test_s, y_test)


**Nộp kèm bài 3**

1. Có bao nhiêu người trong tập test? Tỉ lệ lớp `0` và `1` trên toàn bộ dữ liệu là bao nhiêu?
2. Accuracy cao nhất và `K` tương ứng?
3. Vì sao với bài 2 lớp nên chọn `K` lẻ?


In [ ]:
# Bài 3
# 1. 
# - Số người trong tập test: 
#   + Tùy theo tỷ lệ chia trong hàm chia_train_test (thường là 20% hoặc 30% của 303 dòng):
#   + Nếu test 20%: có khoảng 61 người (len(y_test) = 61).
#   + Nếu test 30%: có khoảng 91 người (len(y_test) = 91).
#   (Bạn có thể kiểm tra trực tiếp bằng lệnh: print(len(y_test))).
# - Tỉ lệ lớp 0 và 1 trên toàn bộ dữ liệu (tổng 303 dòng):
#   + Lớp 0: 138 người (~45.54%)
#   + Lớp 1: 165 người (~54.46%)
#   -> Dữ liệu phân bố tương đối cân bằng (khoảng 45.5% vs 54.5%).

# 2. 
# - Accuracy cao nhất và K tương ứng:
#   + Accuracy cao nhất thường đạt trong khoảng ~82% - 87%.
#   + Thường đạt đỉnh tại các giá trị K lẻ như K = 7, K = 9 hoặc K = 11 (tùy vào cách chia tập train/test ngẫu nhiên).
#   (Bạn nhìn đỉnh cao nhất trên đồ thị `ve_accuracy_theo_K` của bài này để ghi chính xác giá trị K và accuracy tương ứng).

# 3. 
# - Với bài toán phân loại 2 lớp (nhị phân), nên chọn K lẻ vì:
#   + Tránh hiện tượng hòa phiếu (tie-breaking) khi thực hiện biểu quyết đa số (majority vote).
#   + Nếu chọn K chẵn (ví dụ K=4 hoặc K=6), hoàn toàn có khả năng xảy ra trường hợp số láng giềng thuộc lớp 0 và lớp 1 bằng nhau (2 vs 2 hoặc 3 vs 3). Khi đó mô hình sẽ khó đưa ra quyết định dự đoán nhãn nào hoặc phải chọn ngẫu nhiên, làm giảm tính ổn định và độ tin cậy của mô hình.


---

# Bài 4 — Mushroom Classification

Nguồn: https://www.kaggle.com/datasets/uciml/mushroom-classification

File `data/mushrooms.csv`. Đoán nấm ăn được (`e`) hay độc (`p`).

Mọi cột đều là chữ, cùng kiểu demo xe. Cột 0 là nhãn `class`. Các cột còn lại là feature.

- Bỏ dòng nào còn giá trị `?`.
- `LabelEncoder` **từng cột** feature, rồi mã hóa riêng cột nhãn. Giống đoạn mã hóa trong demo xe.
- **Không** chuẩn hóa min–max. Mã `LabelEncoder` chỉ là tên nhóm được đánh số, không phải số đo cùng đơn vị.
- Tập này vài nghìn dòng. `tinh_khoang_cach` cần tính một lần cho cả `x_train` (vector), không viết vòng `for` từng dòng train.


In [ ]:
raw_mush = np.genfromtxt('data/mushrooms.csv', delimiter=',', dtype=str, skip_header=1)
print('trước khi bỏ dòng có ?:', raw_mush.shape)

# TODO: bỏ dòng chứa '?', X_mush float, Y_mush int
# 1. Bỏ các dòng có chứa dấu '?'
mask = ~np.any(raw_mush == '?', axis=1)
mush_clean = raw_mush[mask]

# 2. Mã hóa các cột đặc trưng (từ cột 1 trở đi) thành dạng số float (Label Encoding từng cột)
X_cols = []
for j in range(1, mush_clean.shape[1]):
    _, col_encoded = np.unique(mush_clean[:, j], return_inverse=True)
    X_cols.append(col_encoded)

X_mush = np.column_stack(X_cols).astype(float)

# 3. Mã hóa cột nhãn (cột 0) thành số nguyên int (0: 'e', 1: 'p')
_, Y_mush = np.unique(mush_clean[:, 0], return_inverse=True)

print(X_mush.shape, Y_mush.shape)
print(np.unique(Y_mush, return_counts=True))


In [ ]:
x_train, y_train, x_test, y_test = chia_train_test(X_mush, Y_mush)

# Không gọi chuan_hoa_minmax
print('accuracy K=5:', accuracy(du_doan_tap(x_test, x_train, y_train, 5), y_test))
ve_accuracy_theo_K(x_train, y_train, x_test, y_test)


**Nộp kèm bài 4**

1. Còn bao nhiêu dòng sau khi bỏ `?`? Bao nhiêu feature?
2. Accuracy cao nhất và `K` tương ứng?
3. Bài này giống demo tình trạng xe ở bước nào, và khác bài 1–3 ở chỗ nào (chuẩn hóa)?


In [ ]:
# Bài 4
# 1. 
# - Sau khi bỏ các dòng chứa '?', còn lại 5644 dòng (dữ liệu ban đầu có 8124 dòng, loại bỏ 2480 dòng chứa '?' ở cột stalk-root).
# - Có 22 features (các cột đặc trưng mô tả nấm từ cột 1 đến cột 22, sau khi bỏ cột nhãn class ở cột 0).

# 2. 
# - Accuracy cao nhất thường đạt 1.0 (100%).
# - K tương ứng: Thường đạt 100% tại các K nhỏ như K = 1, 3, 5 (do dữ liệu nấm sau khi mã hóa có tính phân tách rất rõ ràng).
#   (Bạn quan sát đỉnh cao nhất trên đồ thị vẽ accuracy theo K của bài này để ghi chính xác giá trị K).

# 3. 
# - Giống demo tình trạng xe (Car Evaluation):
#   + Đều là dữ liệu phân loại định tính (categorical data/dạng chuỗi chữ), cần phải qua bước mã hóa (Label Encoding / Ordinal Encoding) để chuyển các giá trị chữ sang dạng số (0, 1, 2,...) trước khi đưa vào tính toán.
# - Khác bài 1–3 ở chỗ:
#   + Bài 1–3: Dữ liệu gốc vốn là số thực liên tục (continuous numerical data) đo lường thực tế (chiều dài, cân nặng, nhịp tim,...). Việc chuẩn hóa Min-Max nhằm đưa các giá trị có khoảng chênh lệch thang đo lớn (scale khác nhau) về cùng đoạn [0, 1].
#   + Bài 4: Các con số được tạo ra từ việc gán nhãn định danh (chỉ là mã đại diện cho từng phân loại chữ cái), không phản ánh độ lớn hay khoảng cách liên tục thực tế như bài 1–3. Do đó, bước chuẩn hóa Min-Max ở đây chỉ đơn thuần là ép các mã phân loại rời rạc về khoảng [0, 1].
